# alertness — 学習ノート

特徴量CSVから `model.pkl` を作る。正準ラベルは2軸（drowsiness / distraction）。
Colab で開いたら、上のセルから順に実行する。アルゴリズムは `ALGO` を1行変えるだけ。

In [ ]:
# Colab で開いたら最初に実行：リポジトリを取得して依存を入れ、その中に移動する。
# （Colab は .ipynb だけ読み込むので、コード本体は clone しないと無い）
import os

if not os.path.isdir('training'):            # repo ルートに居なければ取得して移動
    if not os.path.isdir('alertness-colab'):
        !git clone https://github.com/mogmog-0110/alertness-colab.git
    os.chdir('alertness-colab')
!pip install -q -r requirements.txt
print('cwd:', os.getcwd())

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('.'))

import algorithms
import make_dummy_data
from training.config import TrainConfig
from training.metrics import format_scorecard
from training.pipeline import run_training

In [ ]:
# 実データが無ければ仮データを生成（実データは Drive をマウントして DATA_DIR を差し替え）
DATA_DIR = 'sample_data'
if not os.path.isdir(DATA_DIR):
    make_dummy_data.generate(DATA_DIR)

In [ ]:
# ここだけ差し替える。algorithms/ にある同名ファイルが使われる。
ALGO = 'dummy'

# 用途で分けるとき用途名を入れる（'' なら全用途をプール）。
# 眠気は用途に関係なく全データで学習し、注意逸脱だけがこの用途で絞られる。
CONTEXT = ''

In [ ]:
out_path = f'model_{CONTEXT}.pkl' if CONTEXT else 'model.pkl'
cfg = TrainConfig(data_dir=DATA_DIR, context=CONTEXT, out_path=out_path)
result = run_training(cfg, lambda: algorithms.get(ALGO))
print('context:', result['context'] or '(all)')
for target, score in result['scores'].items():
    print(f'=== {target} ===')
    print(format_scorecard(score))
print('saved:', result['artifact'])

## アルゴリズムを足す
1. `algorithms/_template.py` をコピーして `algorithms/名前.py` を作る
2. その `build()` に scikit-learn 互換モデルを返す処理を書く
3. 上の `ALGO = '名前'` にして実行

1ファイル＝1アルゴリズムなので、ファイルを取り合わずに分担できる。

## 用途で分ける
`CONTEXT` に用途名（例 `'study'`）を入れると、注意逸脱はその用途のフレームだけで学習し、
`model_study.pkl` として保存する。眠気は用途に関係なく全データで学習する（用途で意味が
変わらないため。`training/config.py` の `CONTEXT_FREE_AXES`）。用途を分けるほど1モデル
あたりのデータは減るので、データが少ないうちは `CONTEXT=''`（全プール）で始めるのがよい。
データを差し替えるときは `DATA_DIR` を実データのフォルダにする（CSVの列がアプリと同じであればよい）。